# Notebook 01 – Data Collection
## Echo Chambers in Climate Change Discussions on Reddit

This notebook downloads comments and posts from climate-related subreddits
using the **Arctic Shift** public API (no API key needed).

### What this notebook does:
1. Configures which subreddits and date range to collect
2. Downloads comments page-by-page from the Arctic Shift API
3. Downloads post metadata (needed to resolve parent authors)
4. Saves everything as JSONL files in `data/raw/`

> ⏱️ **Expected time:** 30 min – several hours depending on data size.
> 💾 **Storage:** ~500 MB – 2 GB per subreddit for a 4-year range.
> 🔑 **API key:** Not required for Arctic Shift.

In [ ]:
import sys
sys.path.insert(0, '..')  # so we can import from src/

from src.data_collector import RedditCollector
import os

print("✅ Imports successful")

## Configuration
Edit these variables before running:

In [ ]:
# ─── CONFIGURE HERE ───────────────────────────────────────────────────

SUBREDDITS = [
    "climate",          # Main climate discussion
    "climatechange",    # More activism-focused
    "environment",      # Broader environmental topics
    "Collapse",         # Collapse/doomer community
    # "climateskeptics", # Uncomment for contrarian comparison
]

START_DATE = "2021-01-01"   # YYYY-MM-DD
END_DATE   = "2024-12-31"   # YYYY-MM-DD

RAW_DATA_DIR = "../data/raw"

# Delay between API requests in seconds (be polite!)
REQUEST_DELAY = 1.2

# ──────────────────────────────────────────────────────────────────────

print(f"Subreddits : {SUBREDDITS}")
print(f"Date range : {START_DATE}  →  {END_DATE}")
print(f"Output dir : {RAW_DATA_DIR}")

## Step 1: Test API Connection

In [ ]:
import requests

# Quick ping to verify API is reachable
test_url = "https://arctic-shift.photon-reddit.com/api/comments/search"
test_params = {"subreddit": "climate", "limit": 5, "sort": "asc",
               "after": 1609459200, "before": 1609545600}  # 2021-01-01 to 01-02

resp = requests.get(test_url, params=test_params, timeout=20)
print(f"Status code : {resp.status_code}")
data = resp.json()
print(f"Records returned : {len(data.get('data', []))}")

# Show one example comment
if data.get('data'):
    sample = data['data'][0]
    print("\nSample comment fields:")
    for k, v in list(sample.items())[:10]:
        print(f"  {k:25s} : {str(v)[:80]}")

## Step 2: Download Comments (Main Data)

> ⚠️ **For a quick test**, set `TEST_MODE = True` to only download 1 month of data.

In [ ]:
TEST_MODE = False  # Set True to download only 1 month of data for testing

if TEST_MODE:
    start = "2022-01-01"
    end   = "2022-06-30"
    subs  = ["climate"]   # just 1 subreddit
    print("🔧 TEST MODE: downloading small sample only")
else:
    start = START_DATE
    end   = END_DATE
    subs  = SUBREDDITS

collector = RedditCollector(output_dir=RAW_DATA_DIR, delay=REQUEST_DELAY)
results = collector.collect_all(subs, start=start, end=end)

print("\n✅ Download complete!")
print("Files created:")
for sub, paths in results.items():
    print(f"  r/{sub}")
    for kind, p in paths.items():
        import os
        size_mb = os.path.getsize(p) / 1e6 if os.path.exists(p) else 0
        print(f"    {kind}: {p} ({size_mb:.1f} MB)")

## Step 3: Verify Downloaded Files

In [ ]:
import glob, json

print("Files in data/raw/:")
for fpath in sorted(glob.glob(f"{RAW_DATA_DIR}/*.jsonl")):
    # Count lines
    with open(fpath) as f:
        line_count = sum(1 for _ in f)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f"  {os.path.basename(fpath):55s}  {line_count:>8,} records  {size_mb:6.1f} MB")

## Done!
All raw data is saved. Next step: **`02_cleaning.ipynb`** to clean and process it.